In [1]:
from pathlib import Path

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

print("Train file exists:", train_path.exists())
print("Test file exists:", test_path.exists())
print("Sample submission exists:", submission_path.exists())

Train file exists: True
Test file exists: True
Sample submission exists: True


In [2]:
import pandas as pd
import pyarrow.parquet as pq

test_file = pq.ParquetFile(test_path)
sample_submission = pd.read_csv(submission_path)

print("Test spectra:", test_file.metadata.num_rows)
print("Test columns:", test_file.schema_arrow.names)
print("Submission columns:", sample_submission.columns.tolist())
print("Sample submission rows:", len(sample_submission))

Test spectra: 1213
Test columns: ['molecule_id', 'spectrum_id', 'ms2_mzs', 'ms2_normalized_intensities', 'base_peak_intensity', 'adduct', 'ionization_mode', 'instrument_type', 'precursor_mz', 'collision_energy_orig', 'collision_energy_ev', 'collision_energy_orig_units']
Submission columns: ['molecule_id', 'smiles']
Sample submission rows: 400


In [3]:
# CASMI 2026 - Prepare test spectra for exact matching

import hashlib
import numpy as np
from collections import defaultdict

test_columns = [
    "molecule_id",
    "spectrum_id",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

test_spectra = test_file.read(
    columns=test_columns
).to_pandas()


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    fingerprint = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), fingerprint)


# Connect each fingerprint to its test molecule and spectrum
test_lookup = defaultdict(list)

for row in test_spectra.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)

    test_lookup[fingerprint].append(
        (row.molecule_id, row.spectrum_id)
    )


print("CASMI 2026 - Test Spectra Prepared")
print("----------------------------------")
print("Test spectra:", len(test_spectra))
print(
    "Unique test molecules:",
    test_spectra["molecule_id"].nunique()
)
print("Unique spectrum fingerprints:", len(test_lookup))

print(
    "Submission molecule IDs match test data:",
    set(sample_submission["molecule_id"])
    == set(test_spectra["molecule_id"])
)

print("\nTest spectra prepared successfully!")

CASMI 2026 - Test Spectra Prepared
----------------------------------
Test spectra: 1213
Unique test molecules: 400
Unique spectrum fingerprints: 1213
Submission molecule IDs match test data: True

Test spectra prepared successfully!


In [4]:
# CASMI 2026 - Find exact matches in the training dataset

import pyarrow.parquet as pq
from collections import defaultdict

train_file = pq.ParquetFile(train_path)

train_columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Skip spectra with adduct/peak-count combinations
# that do not occur in the test data.
test_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in test_spectra.itertuples(index=False)
}

# molecule_id -> (inchikey14, SMILES) -> matched spectrum IDs
exact_hits = defaultdict(lambda: defaultdict(set))

print("CASMI 2026 - Exact Match Search")
print("-------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=10000,
        columns=train_columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            combination = (
                row.adduct,
                len(row.ms2_mzs)
            )

            if combination not in test_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id, spectrum_id in test_lookup.get(
                fingerprint, []
            ):
                exact_hits[molecule_id][
                    (row.inchikey14, row.normalized_smiles)
                ].add(spectrum_id)

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Summarize without assuming that all test molecules matched
test_molecule_count = test_spectra["molecule_id"].nunique()

matched_molecules = len(exact_hits)

unique_matches = sum(
    len(structures) == 1
    for structures in exact_hits.values()
)

ambiguous_matches = sum(
    len(structures) > 1
    for structures in exact_hits.values()
)

print("\nExact matching results:")
print("Test molecules:", test_molecule_count)
print("Molecules with exact matches:", matched_molecules)
print("Molecules with one matched structure:", unique_matches)
print("Molecules with multiple matched structures:", ambiguous_matches)
print(
    "Molecules without an exact match:",
    test_molecule_count - matched_molecules
)

CASMI 2026 - Exact Match Search
-------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

Exact matching results:
Test molecules: 400
Molecules with exact matches: 400
Molecules with one matched structure: 400
Molecules with multiple matched structures: 0
Molecules without an exact match: 0


In [5]:
# CASMI 2026 - Prepare exact-match predictions

exact_predictions = {}
unresolved_molecules = []

for molecule_id in sample_submission["molecule_id"]:

    matched_structures = exact_hits.get(molecule_id, {})

    # Use an exact match only when it identifies one structure
    if len(matched_structures) == 1:

        inchikey14, smiles = next(
            iter(matched_structures.keys())
        )

        if (
            isinstance(smiles, str)
            and smiles.strip()
            and ";" not in smiles
        ):
            exact_predictions[molecule_id] = smiles
        else:
            unresolved_molecules.append(molecule_id)

    else:
        unresolved_molecules.append(molecule_id)


print("CASMI 2026 - Exact Predictions")
print("------------------------------")
print("Submission molecules:", len(sample_submission))
print("Exact predictions:", len(exact_predictions))
print("Unresolved molecules:", len(unresolved_molecules))

print("\nFirst 3 exact predictions:")
for molecule_id, smiles in list(exact_predictions.items())[:3]:
    print(molecule_id, "->", smiles)

print("\nExact prediction mapping prepared!")

CASMI 2026 - Exact Predictions
------------------------------
Submission molecules: 400
Exact predictions: 400
Unresolved molecules: 0

First 3 exact predictions:
m_005e53 -> CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1
m_006153 -> COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1
m_00b5aa -> COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC

Exact prediction mapping prepared!


In [6]:
# CASMI 2026 - Build a fallback candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

for batch in train_file.iter_batches(
    batch_size=20000,
    columns=library_columns
):
    batch_df = batch.to_pandas()

    # Remove incomplete rows and repeated molecular structures
    batch_df = batch_df.dropna(
        subset=library_columns
    ).drop_duplicates(subset="inchikey14")

    new_rows = batch_df[
        ~batch_df["inchikey14"].isin(seen_keys)
    ]

    if not new_rows.empty:
        library_parts.append(new_rows)
        seen_keys.update(new_rows["inchikey14"])

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

print("CASMI 2026 - Fallback Candidate Library")
print("---------------------------------------")
print("Unique molecular structures:", len(full_candidate_library))
print(
    "Missing SMILES:",
    full_candidate_library["normalized_smiles"].isna().sum()
)
print(
    "Missing molecular formulas:",
    full_candidate_library["molecular_formula"].isna().sum()
)

print("\nCandidate library prepared!")

CASMI 2026 - Fallback Candidate Library
---------------------------------------
Unique molecular structures: 275810
Missing SMILES: 0
Missing molecular formulas: 0

Candidate library prepared!


In [7]:
# CASMI 2026 - Calculate candidate reference masses

import re
import numpy as np

# Monoisotopic atomic masses in Da
atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    # Remove trailing charge notation, where present
    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    # Reject formulas we cannot parse completely
    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )


full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Mass Calculation")
print("---------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

print("\nReference mass calculation completed!")

CASMI 2026 - Reference Mass Calculation
---------------------------------------
Total candidates: 275810
Candidates with reference mass: 275763
Candidates with missing reference mass: 47

Reference mass calculation completed!


In [8]:
# CASMI 2026 - Calculate neutral masses for test molecules

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

# Read the test metadata available during this notebook run
mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(mass_data["precursor_mz"], errors="coerce")
    - mass_data["adduct_shift"]
)

# Calculate one median neutral mass per molecule
test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

print("CASMI 2026 - Test Neutral Masses")
print("--------------------------------")
print("Test spectra:", len(mass_data))
print("Test molecules:", len(test_molecules))

print(
    "Spectra with unknown adducts:",
    int(mass_data["adduct_shift"].isna().sum())
)

print(
    "Molecules with missing neutral mass:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)

print("\nFirst 5 molecules:")
print(test_molecules.head().to_string(index=False))

CASMI 2026 - Test Neutral Masses
--------------------------------
Test spectra: 1213
Test molecules: 400
Spectra with unknown adducts: 0
Molecules with missing neutral mass: 0

First 5 molecules:
molecule_id  median_neutral_mass  num_spectra
   m_005e53           374.063624            4
   m_006153           341.149200            6
   m_00b5aa           306.195024            2
   m_01e922           249.137124            3
   m_0259d4           339.170224            7


In [9]:
# CASMI 2026 - Generate submission.csv

import numpy as np
import pandas as pd
from pathlib import Path

# Prepare the mass-based candidate library
mass_library = (
    full_candidate_library
    .dropna(subset=["reference_mass", "normalized_smiles"])
    .sort_values("reference_mass")
    .reset_index(drop=True)
)

sorted_masses = mass_library["reference_mass"].to_numpy(
    dtype=float
)
sorted_smiles = mass_library["normalized_smiles"].to_numpy()

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

submission_rows = []

for molecule_id in sample_submission["molecule_id"]:

    selected_smiles = []
    seen_smiles = set()

    # Place an unambiguous exact match first, if available
    exact_smiles = exact_predictions.get(molecule_id)

    if exact_smiles is not None:
        selected_smiles.append(exact_smiles)
        seen_smiles.add(exact_smiles)

    # Fill remaining positions using nearby reference masses
    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        raise ValueError(
            f"No usable neutral mass for {molecule_id}"
        )

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(
                sorted_masses[left] - query_mass
            )
            right_error = abs(
                sorted_masses[right] - query_mass
            )

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(selected_smiles)
    })

submission = pd.DataFrame(submission_rows)

# Check the required format
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert len(submission) == len(sample_submission)

assert (
    submission["molecule_id"].tolist()
    == sample_submission["molecule_id"].tolist()
)

assert all(
    len(smiles.split(";")) == 25
    and len(set(smiles.split(";"))) == 25
    for smiles in submission["smiles"]
)

# Save using the filename required by Kaggle
output_path = Path("/kaggle/working/submission.csv")
submission.to_csv(output_path, index=False)

print("CASMI 2026 - Submission Created")
print("--------------------------------")
print("File exists:", output_path.exists())
print("File path:", output_path)
print("Submission rows:", len(submission))
print("SMILES per molecule: 25")
print(
    "Molecules with exact prediction:",
    sum(
        molecule_id in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print(
    "Molecules using mass-only fallback:",
    sum(
        molecule_id not in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print("\nSubmission format validation passed!")

CASMI 2026 - Submission Created
--------------------------------
File exists: True
File path: /kaggle/working/submission.csv
Submission rows: 400
SMILES per molecule: 25
Molecules with exact prediction: 400
Molecules using mass-only fallback: 0

Submission format validation passed!
